# NB08 — Machine Learning: Return Direction & Price Forecasting

**Task A — Return Direction Classification** (binary: up/down next 5 days):
- Models: Logistic Regression, Random Forest, XGBoost, SVM (RBF kernel)
- Metrics: Accuracy, Precision, Recall, F1, AUC-ROC, Brier Score
- Threshold optimization: move decision boundary to maximize Sharpe
- Calibration: Platt scaling or isotonic regression

**Task B — Return Magnitude Regression** (continuous: 5-day forward return):
- Same model suite as NB07 (Ridge, RF, XGBoost, LightGBM)

**Task C — Multi-Horizon Forecasting** (1d, 5d, 21d):
- Performance decay analysis across horizons

**Walk-Forward Protocol**: Same as NB07 — expanding window, quarterly retraining.

**CRITICAL**: SMOTE used ONLY for classification. 10 bps transaction costs applied.

**Dependencies**: NB01 (master data), NB03 (conditional vol), NB05 (regimes), NB06 (sentiment)

**Output**: `return_predictions.parquet`, `classification_report.csv`, `strategy_backtest.csv`

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import *
from src.feature_engineering import (
    compute_log_returns, realized_vol, rsi, bollinger_pct_b,
    rate_of_change, volume_zscore, rolling_beta
)
from src.ml_pipeline import (
    make_regression_pipeline, make_classification_pipeline,
    walk_forward_predict, regression_metrics, classification_metrics,
    diebold_mariano_test
)
from src.visualization import save_fig
import logging
logging.basicConfig(level=logging.INFO)
print('Imports OK')

Imports OK


## 1. Load Data & Build Features

In [2]:
# Load upstream data
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
log_ret = compute_log_returns(master[adj_tickers])
print(f'Master: {master.shape}, Tickers: {len(adj_tickers)}')

# Optional upstream features
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else pd.DataFrame()
regime = pd.read_parquet(REGIME_LABELS_FILE) if REGIME_LABELS_FILE.exists() else pd.DataFrame()
sentiment = pd.read_parquet(SENTIMENT_FILE) if SENTIMENT_FILE.exists() else pd.DataFrame()

spy_col = [c for c in master.columns if 'SPY' in c]
spy_ret = compute_log_returns(master[[spy_col[0]]])[spy_col[0]] if spy_col else None

def build_return_features(ticker):
    """Build feature matrix for return forecasting (same feature set as NB07)."""
    prices = master[ticker].dropna()
    log_r = np.log(prices / prices.shift(1))

    X = pd.DataFrame(index=prices.index)
    for w in [5, 21, 63]:
        X[f'realized_vol_{w}d'] = log_r.rolling(w).std() * np.sqrt(252)
    X['log_return'] = log_r
    X['abs_return'] = log_r.abs()
    X['return_5d'] = prices.pct_change(5)
    X['return_21d'] = prices.pct_change(21)
    X['rsi_14'] = rsi(prices, 14)
    X['bollinger_pctb'] = bollinger_pct_b(prices)
    X['roc_20d'] = rate_of_change(prices, 20)
    X['vol_ratio_5_21'] = X['realized_vol_5d'] / X['realized_vol_21d'].clip(lower=1e-6)

    if not cond_vol.empty and ticker in cond_vol.columns:
        X['garch_cond_vol'] = cond_vol[ticker].reindex(prices.index)
    if spy_ret is not None:
        X['beta_spy_63d'] = rolling_beta(log_r, spy_ret, window=63)
    if not regime.empty and 'regime_state' in regime.columns:
        X['regime_state'] = regime['regime_state'].reindex(prices.index)
    if not sentiment.empty:
        ticker_sent = sentiment[sentiment['ticker'] == ticker].drop(columns=['ticker'], errors='ignore')
        for col in ticker_sent.columns:
            X[f'sent_{col}'] = ticker_sent[col].reindex(prices.index)

    return X.dropna(), prices

print('Feature builder ready')

Master: (2526, 33), Tickers: 20
Feature builder ready


In [ ]:
# Task A: Return Direction Classification (all tickers × 4 models)
CLF_MODELS = ['logistic', 'random_forest', 'xgboost', 'svm']
clf_results = []
clf_predictions = {}

for ticker in adj_tickers:
    X, prices = build_return_features(ticker)
    fwd_5d = prices.pct_change(5).shift(-5)
    direction = (fwd_5d > 0).astype(int)
    y = direction.reindex(X.index)
    valid = ~(X.isna().any(axis=1) | y.isna())
    X_clf, y_clf = X[valid], y[valid]

    if len(X_clf) < 200:
        continue

    for model_name in CLF_MODELS:
        try:
            preds = walk_forward_predict(
                X_clf, y_clf,
                pipeline_factory=lambda n=model_name: make_classification_pipeline(n, use_smote=True),
                retrain_freq=RETRAIN_FREQ_DAYS, initial_train_ratio=TRAIN_RATIO,
                task='classification',
                horizon=5  # 5-day forward target → purge/embargo 5 days
            )
            if len(preds) > 0:
                y_prob = preds.get('y_prob', None)
                metrics = classification_metrics(
                    preds['y_true'].values, preds['y_pred'].values,
                    y_prob.values if y_prob is not None else None
                )
                metrics['ticker'] = ticker
                metrics['model'] = model_name
                metrics['n_predictions'] = len(preds)
                clf_results.append(metrics)
                clf_predictions[(ticker, model_name)] = preds
        except Exception as e:
            continue

    print(f'{ticker}: {len([r for r in clf_results if r["ticker"]==ticker])} classifiers done')

clf_df = pd.DataFrame(clf_results)
print(f'\nTotal: {len(clf_df)} ticker-model combinations')

# Average classification metrics per model
clf_avg = clf_df.groupby('model')[['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'brier_score']].mean()
clf_avg = clf_avg.sort_values('f1', ascending=False)
print('\n--- Average Classification Metrics ---')
print(clf_avg.round(3))

# Best classifier per ticker
best_clf_per_ticker = clf_df.loc[clf_df.groupby('ticker')['f1'].idxmax()]
print(f'\nClassifier wins: {best_clf_per_ticker["model"].value_counts().to_dict()}')

## 2. Task B — Return Magnitude Regression (All Tickers)

No SMOTE here — this is REGRESSION. Predict the actual 5-day forward return.

In [ ]:
# Task B: Return Magnitude Regression (all tickers × 4 models)
REG_MODELS = ['ridge', 'random_forest', 'xgboost', 'lightgbm']
reg_results = []
reg_predictions = {}

for ticker in adj_tickers:
    X, prices = build_return_features(ticker)
    fwd_5d_ret = prices.pct_change(5).shift(-5)
    y = fwd_5d_ret.reindex(X.index)
    valid = ~(X.isna().any(axis=1) | y.isna())
    X_reg, y_reg = X[valid], y[valid]

    if len(X_reg) < 200:
        continue

    for model_name in REG_MODELS:
        try:
            preds = walk_forward_predict(
                X_reg, y_reg,
                pipeline_factory=lambda n=model_name: make_regression_pipeline(n),
                retrain_freq=RETRAIN_FREQ_DAYS, initial_train_ratio=TRAIN_RATIO,
                horizon=5  # 5-day forward target → purge/embargo 5 days
            )
            if len(preds) > 0:
                metrics = regression_metrics(preds['y_true'].values, preds['y_pred'].values)
                metrics['ticker'] = ticker
                metrics['model'] = model_name
                reg_results.append(metrics)
                reg_predictions[(ticker, model_name)] = preds
        except Exception as e:
            continue

    print(f'{ticker}: regression done')

reg_df = pd.DataFrame(reg_results)
reg_avg = reg_df.groupby('model')[['rmse', 'mae', 'directional_accuracy']].mean().sort_values('rmse')
print('\n--- Return Regression: Average Metrics ---')
print(reg_avg.round(4))

## 3. Task C — Multi-Horizon Performance Decay

How quickly does forecast skill degrade with horizon?
Test at 1d, 5d, 21d using Ridge (fastest baseline) across all tickers.

In [ ]:
# Use a representative ticker explicitly (not leftover from loop)
ticker_mh = 'NVDA'
X_mh, prices_mh = build_return_features(ticker_mh)

for horizon in [1, 5, 21]:
    y_h = prices_mh.pct_change(horizon).shift(-horizon).reindex(X_mh.index)
    valid_h = ~(X_mh.isna().any(axis=1) | y_h.isna())
    if valid_h.sum() < 100: continue
    preds = walk_forward_predict(X_mh[valid_h], y_h[valid_h],
        pipeline_factory=lambda: make_regression_pipeline('ridge'),
        horizon=horizon)  # match horizon for correct purge/embargo
    if len(preds) > 0:
        m = regression_metrics(preds['y_true'].values, preds['y_pred'].values)
        print(f'{ticker_mh} Horizon {horizon}d: RMSE={m["rmse"]:.6f}, DA={m["directional_accuracy"]:.1f}%')

## 4. Task D — Cross-Sectional Momentum & Microstructure

- **Cross-sectional momentum**: 12-1 month signal ranked across 20 tickers
- **Information Coefficient**: Spearman rank corr between signal and forward return
- **Engle-Granger cointegration** for 5 natural pairs
- **Lead-lag cross-correlation**: identify if any ticker leads others

In [ ]:
from src.feature_engineering import (
    cross_sectional_momentum, engle_granger_cointegration,
    lead_lag_crosscorr, information_coefficient
)

prices = master[adj_tickers].dropna(how='all')

# ── 4.1 Cross-Sectional Momentum (12-1 month) ──
mom_signal = cross_sectional_momentum(prices, lookback=252, skip=22)
print(f'Momentum signal shape: {mom_signal.shape}')
print(f'Latest momentum rankings:')
latest_mom = mom_signal.iloc[-1].sort_values(ascending=False)
for t, v in latest_mom.head(5).items():
    print(f'  {t}: {v:.4f}')

# ── 4.2 Information Coefficient ──
fwd_ret_21d = prices.pct_change(21).shift(-21)
ic_values = []
for date in mom_signal.index[::21]:  # monthly IC
    if date not in fwd_ret_21d.index:
        continue
    signal_row = mom_signal.loc[date].dropna()
    fwd_row = fwd_ret_21d.loc[date].reindex(signal_row.index).dropna()
    common = signal_row.index.intersection(fwd_row.index)
    if len(common) >= 5:
        ic = information_coefficient(signal_row.loc[common], fwd_row.loc[common])
        ic_values.append(ic)

print(f'\n--- Information Coefficient (Momentum → 21d Forward Return) ---')
print(f'  Mean IC:   {np.mean(ic_values):.4f} (>0.05 is economically meaningful)')
print(f'  IC Std:    {np.std(ic_values):.4f}')
print(f'  IC > 0:    {(np.array(ic_values) > 0).mean():.1%} of periods')
print(f'  IR (IC/σ): {np.mean(ic_values)/np.std(ic_values):.3f}')

# ── 4.3 Engle-Granger Cointegration (5 pairs) ──
pairs = [('NVDA', 'AMD'), ('TSM', 'AVGO'), ('META', 'GOOG'), ('PANW', 'CRWD'), ('CRM', 'NOW')]
coint_results = []
for t1, t2 in pairs:
    if t1 in prices.columns and t2 in prices.columns:
        result = engle_granger_cointegration(prices[t1].dropna(), prices[t2].dropna())
        result['pair'] = f'{t1}-{t2}'
        coint_results.append(result)
        print(f'{t1}-{t2}: coint_p={result["p_value"]:.4f}, '
              f'hedge_ratio={result["beta"]:.3f}, '
              f'half_life={result["half_life"]:.1f}d')

coint_df = pd.DataFrame(coint_results)
coint_df.to_csv(TABLES_DIR / 'nb08_cointegration_pairs.csv', index=False)

# ── 4.4 Lead-Lag Cross-Correlation ──
log_ret = np.log(prices / prices.shift(1)).dropna()
leadlag_results = []
test_pairs = [('NVDA', 'AMD'), ('TSM', 'NVDA'), ('NVDA', 'AVGO'), ('META', 'GOOG')]
for t1, t2 in test_pairs:
    if t1 in log_ret.columns and t2 in log_ret.columns:
        xcorr = lead_lag_crosscorr(log_ret[t1], log_ret[t2], max_lag=5)
        max_lag_idx = xcorr.abs().idxmax()
        leadlag_results.append({
            'pair': f'{t1}-{t2}', 'max_corr_lag': max_lag_idx,
            'max_corr': xcorr.loc[max_lag_idx],
            'lag0_corr': xcorr.loc[0]
        })
        if max_lag_idx != 0:
            leader = t1 if max_lag_idx > 0 else t2
            print(f'{t1}-{t2}: max corr at lag {max_lag_idx} ({leader} leads)')

ll_df = pd.DataFrame(leadlag_results)
ll_df.to_csv(TABLES_DIR / 'nb08_lead_lag_network.csv', index=False)
print(f'\nSaved: cointegration_pairs.csv, lead_lag_network.csv')

## 5. Save Outputs

Persist all predictions and metrics for downstream notebooks (NB10, NB11):
- `return_predictions.parquet` — walk-forward regression predictions per ticker
- `classification_report.csv` — classification metrics per model × ticker
- `strategy_backtest.csv` — simple long/flat strategy with transaction costs

In [ ]:
from src.config import (
    RETURN_PRED_FILE, CLASS_REPORT_FILE, STRATEGY_BACKTEST_FILE,
    TRANSACTION_COST_BPS,
)

# ── 5a. Save classification report ──
if len(clf_df) > 0:
    clf_df.to_csv(CLASS_REPORT_FILE, index=False)
    print(f'Saved classification report: {CLASS_REPORT_FILE}')

# ── 5b. Save return predictions (regression) ──
if reg_predictions:
    all_preds = []
    for (ticker, model_name), preds_df in reg_predictions.items():
        p = preds_df.copy()
        p['ticker'] = ticker
        p['model'] = model_name
        all_preds.append(p)
    pred_df = pd.concat(all_preds, ignore_index=True)
    pred_df.to_parquet(RETURN_PRED_FILE)
    print(f'Saved return predictions: {RETURN_PRED_FILE}  ({len(pred_df)} rows)')

# ── 5c. Strategy backtest: simple long/flat from best classifier ──
# Use best classifier (by avg F1) to generate daily long/flat signals
tc_daily = TRANSACTION_COST_BPS / 10_000  # 10 bps round-trip

strat_rows = []
if clf_predictions:
    # Pick the best model overall
    best_model = clf_avg.index[0]  # highest F1
    for ticker in adj_tickers:
        key = (ticker, best_model)
        if key not in clf_predictions:
            continue
        preds = clf_predictions[key].copy()
        if 'y_pred' not in preds.columns or 'y_true' not in preds.columns:
            continue
        # Signal: go long when predicted up (1), flat when down (0)
        preds['signal'] = preds['y_pred'].astype(int)
        # Actual daily returns (use log returns from master)
        tkr_rets = log_ret[ticker].reindex(preds.index).fillna(0)
        preds['daily_return'] = tkr_rets
        # Strategy return = signal * return - transaction cost on trades
        preds['trade'] = preds['signal'].diff().abs().fillna(0)
        preds['strategy_return'] = (
            preds['signal'] * preds['daily_return'] - preds['trade'] * tc_daily
        )
        preds['cum_strategy'] = (1 + preds['strategy_return']).cumprod()
        preds['cum_buyhold'] = (1 + preds['daily_return']).cumprod()
        total_strat = preds['cum_strategy'].iloc[-1] - 1
        total_bh = preds['cum_buyhold'].iloc[-1] - 1
        n_days = len(preds)
        sharpe = (
            preds['strategy_return'].mean() / preds['strategy_return'].std()
            * np.sqrt(252)
        ) if preds['strategy_return'].std() > 0 else 0
        strat_rows.append({
            'ticker': ticker, 'model': best_model, 'n_days': n_days,
            'strategy_total_return': total_strat,
            'buyhold_total_return': total_bh,
            'strategy_sharpe': sharpe,
            'n_trades': int(preds['trade'].sum()),
            'tc_total_bps': preds['trade'].sum() * TRANSACTION_COST_BPS,
        })

strat_df = pd.DataFrame(strat_rows)
if len(strat_df) > 0:
    strat_df.to_csv(STRATEGY_BACKTEST_FILE, index=False)
    print(f'Saved strategy backtest: {STRATEGY_BACKTEST_FILE}')
    print(f'\n--- Long/Flat Strategy ({best_model}) ---')
    print(strat_df[['ticker', 'strategy_total_return', 'buyhold_total_return',
                     'strategy_sharpe', 'n_trades']].round(4).to_string(index=False))
else:
    print('No strategy backtest results to save')
